# Train LeWM on OGBCubeDR (RunPod)

Trains LeWM on `cube_quadruple_dr_expert` (5000 `OGBCubeDR-v0` episodes, see `Run.md`). Flow: config → apply → torch check → install → download → sanity check → train.

Prerequisite: dataset already pushed to HF via `upload_to_hf.sh` (repo root).

## 1. Config

Edit the values below. `STABLEWM_HOME=/workspace` puts the dataset at `/workspace/datasets/ogbench/...`. `HF_TOKEN`/`WANDB_API_KEY` come from pod env vars if set, else paste them in.

In [ ]:
import os

# --- repo ---
REPO_ROOT = '/workspace/stable-worldmodel'          # ← edit if you cloned it elsewhere

# --- storage (network volume) ---
STABLEWM_HOME = '/workspace'                        # datasets/, checkpoints/ land directly here
SPT_CACHE_DIR = '/workspace/cache/stable-pretraining'  # Lightning .ckpt files (see note below)

# --- Hugging Face ---
HF_TOKEN = os.environ.get('HF_TOKEN', '')                          # ← paste here if not set as a pod env var
HF_REPO_ID = '<your-hf-username-or-org>/ogbench-cube-quadruple-domain-randomized-expert'  # ← edit me

# --- Weights & Biases ---
WANDB_API_KEY = os.environ.get('WANDB_API_KEY', '')  # ← paste here if not set as a pod env var
WANDB_ENTITY = '<your-wandb-entity>'                 # ← edit me
WANDB_PROJECT = 'ogbcubedr-lewm'                     # ← edit me if you want a different project name

# --- training run naming ---
OUTPUT_MODEL_NAME = 'lewm_q4_dr'

# --- derived, don't edit ---
DATASET_DIR = os.path.join(STABLEWM_HOME, 'datasets', 'ogbench', 'cube_quadruple_dr_expert.lance')

## 2. Apply config

In [ ]:
os.environ['STABLEWM_HOME'] = STABLEWM_HOME
os.environ['SPT_CACHE_DIR'] = SPT_CACHE_DIR
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['WANDB_API_KEY'] = WANDB_API_KEY

os.makedirs(STABLEWM_HOME, exist_ok=True)
os.makedirs(SPT_CACHE_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

os.chdir(REPO_ROOT)  # os.chdir (not `%cd`) so it's identical whether run fresh or after a kernel restart

print('cwd           =', os.getcwd())
print('STABLEWM_HOME =', os.environ['STABLEWM_HOME'])
print('SPT_CACHE_DIR =', os.environ['SPT_CACHE_DIR'])
print('DATASET_DIR   =', DATASET_DIR)
!df -h /workspace

## 3. Torch ≥ 2.5

`transformers` needs `torch>=2.5`; some pods ship 2.4.1. Upgrades torch+torchvision+torchaudio together. **Restart the kernel if it upgrades**, then re-run cells 1–2.

In [ ]:
import torch
print('torch before:', torch.__version__)

if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 5):
    !pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
    print('Upgraded — RESTART THE KERNEL now, then re-run cells 1-2 before continuing.')
else:
    print('torch already >= 2.5, nothing to do.')

## 4. Install dependencies

In [ ]:
%pip install -q -e '.[train,format]' wandb huggingface_hub

## 5. Download dataset

Uses `HF_TOKEN` from the config cell. Lands at `$STABLEWM_HOME/datasets/ogbench/cube_quadruple_dr_expert.lance/`. Safe to re-run.

In [ ]:
!hf download "$HF_REPO_ID" --repo-type dataset --local-dir "$DATASET_DIR"

## 6. W&B login

In [ ]:
import wandb

wandb.login(key=WANDB_API_KEY)

## 7. Sanity check

Expect 5000 episodes × 401 steps, and a visible GPU.

In [ ]:
import lance

ds = lance.dataset(DATASET_DIR)
ep = ds.to_table(columns=['episode_idx']).column('episode_idx').to_numpy()
print(f'rows: {ds.count_rows():,}')
print(f'episodes: {ep.max() - ep.min() + 1}')

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU visible — lewm.yaml requires accelerator: gpu'
print(torch.cuda.get_device_name(0))

## 8. Train

Smoke test first, then the full run.

In [ ]:
!python scripts/train/lewm.py data=ogb_cube_quadruple_dr \
    output_model_name="${OUTPUT_MODEL_NAME}_smoke" \
    trainer.max_epochs=1 \
    loader.batch_size=128 loader.num_workers=2 \
    wandb.enabled=false \
    hydra.run.dir=/tmp/lewm_smoke

Full run — launched detached (survives closing the browser/kernel; only dies if the pod stops):

In [ ]:
import subprocess

log_path = os.path.join(STABLEWM_HOME, 'logs', f'{OUTPUT_MODEL_NAME}.log')
os.makedirs(os.path.dirname(log_path), exist_ok=True)

cmd = [
    'python', 'scripts/train/lewm.py',
    'data=ogb_cube_quadruple_dr',
    f'output_model_name={OUTPUT_MODEL_NAME}',
    'wandb.enabled=true',
    f'wandb.config.entity={WANDB_ENTITY}',
    f'wandb.config.project={WANDB_PROJECT}',
]

with open(log_path, 'w') as f:
    proc = subprocess.Popen(
        cmd, stdout=f, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        start_new_session=True,  # detaches from this kernel's process group/session
    )

print('Started PID', proc.pid)
print('Log:', log_path)

Check progress any time (also visible live on the W&B dashboard):

In [ ]:
!tail -n 40 "$log_path"

Checkpoints: `$STABLEWM_HOME/checkpoints/$OUTPUT_MODEL_NAME/weights_epoch_N.pt` (used by `eval_wm.py`). Lightning's `.ckpt`s go to `$SPT_CACHE_DIR/runs/.../checkpoints/` — not used by eval. See `Run.md` §6–7.